# 08 – Model B: grid stress (LightGBM + SHAP)

**Goal:** explain and score which buurten are at risk of grid stress, and why.
**Input:** `Data/processed/model_b_table.csv` (5,423 buurten, snapshot)
**Outputs:** `buurt_stress_predictions.csv`, `shap_importance_util.csv`, `shap_importance_stress.csv`

## Validation
- Snapshot → no time split. **5-fold cross-validation grouped by municipality** (141 municipalities): every buurt is predicted by a model that never saw its municipality (prevents spatial leakage).
- 50 features; nothing computed from the peaks (leakage rule).
- Class imbalance (10% stressed) not corrected with class weights: the model is judged on ranking (ROC-AUC / PR-AUC).
- Stress label = peak **above** the NBL limit (555 buurten). The priority levels in notebook 09 use **at or above** (577 buurten); the 22 extra buurten have a transformer exactly at its limit.

## Results
| Target | Metric | Guessing / average | Size only (3 features) | All features (50) |
|---|---|---|---|---|
| `stress_nbl` | ROC-AUC | 0.50 | 0.684 | **0.753** |
| `stress_nbl` | PR-AUC | 0.102 | 0.200 | **0.271** (2.7× guessing) |
| `util_weighted` | MAE | 0.130 | 0.097 | **0.084** |
| `util_weighted` | R² | 0.00 | 0.369 | **0.502** |
| `util_weighted` | ranking | – | 0.63 | **0.74** |

- A single column `n_transformers` already gives ROC-AUC 0.70 → `stress_nbl` is partly a size effect. Drivers add +0.07 AUC on top.
- For `util_weighted`, "size only" still scores R² 0.37 because capacity + connections = capacity per connection (the core infrastructure ratio).

## Drivers (SHAP)
- **Utilisation:** demand per kW capacity (+), capacity per connection (−), density (+), compact buurten (+), older housing (+), solar feed-in per kW capacity (+, small).
- **Stress label:** dominated by volume/size (total kWh, businesses, number of transformers), then the same demand-pressure drivers.
- Gas-free homes, 3-phase connections and solar homes per transformer go with *less* stress → likely newer areas with a modern grid.
- Caution: SHAP = associations, not causes; correlated features share credit → interpret groups of drivers.

## Driver labels per buurt (corrected after review)
- First version used fixed phrases ("many connections per transformer") that did not follow the SHAP direction → 2,479 buurten had at least one wrong label. The cells that printed those first labels are kept for transparency but are superseded.
- **Fix:** labels say "high"/"low" based on the buurt's real value vs the median of all buurten; hidden CBS values say "unknown (hidden by CBS)". Check: 0 contradictions (re-checked in the review: 0 of 16,163 labels wrong).
- Labels come from the regression SHAP (size-independent), with no duplicate names.
- Most common first driver: high demand per kW capacity (2,170), low buurt area (667), high number of businesses (635), low connections per transformer (619: few but heavy connections, e.g. businesses, farms, homes with large solar systems).

## Solar test (Liander PV per buurt, 2024)
- 42% of buurten have no Liander PV value (hidden for privacy); these are larger, more urban buurten with the same CBS solar share → gap in the data, not in reality.
- Where available: utilisation rises with solar kW per kW capacity (0.34 → 0.51; Spearman 0.36), but more strongly with average consumption per kW (Spearman 0.57). Stress rate falls with solar (12.1% → 4.5%).
- Median peak ≈ 0.97 × installed solar → peaks are larger than solar alone can explain.
- Summer peaks (~98%) occur equally in low- and high-solar buurten → timing is likely a measurement artefact, not evidence of feed-in. To verify with Liander/Deloitte.
- **Conclusion:** stress = peak utilisation from a mix of consumption and feed-in; the data cannot fully separate them. Liander PV not added to the model (non-random missing data); CBS-based solar features kept.

## Risk tiers (out-of-fold probability)
| Tier | Buurten | Actual stress rate |
|---|---|---|
| Low (bottom 50%) | 2,712 | 3.7% |
| Medium | 1,626 | 10.8% |
| High | 813 | 21.8% |
| Very high (top 5%) | 272 | 37.1% |

- Examples at the top: city centres with many businesses (Arnhem, Nijmegen, Ede). A single overloaded transformer can make a large, otherwise well-equipped buurt "stressed" (e.g. Markt, Arnhem: 1 of 21 transformers).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

PROCESSED = Path("../Data/processed")
b = pd.read_csv(PROCESSED / "model_b_table.csv")

# municipality code from the buurt code: BU + GGGG (municipality) + WW + BB
b["gm"] = b["buurtcode"].str[2:6]

# never use anything computed from the peaks (leakage)
leak = ["peak_kw_total", "util_max", "n_above_nbl", "n_above_capacity",
        "share_summer_peaks", "util_weighted", "stress_nbl", "stress_capacity"]
features = [c for c in b.columns if c not in ["buurtcode", "gm"] + leak]
print(f"{len(features)} features:")
print(features)
print(f"\nBuurten: {len(b):,} | municipalities: {b['gm'].nunique()}")

y = b["stress_nbl"]
print(f"Stressed share (= PR-AUC of guessing): {y.mean():.3f}")

# single-feature baselines: how well does ONE column rank stressed buurten?
print("\nSingle-feature baselines:")
for f in ["n_transformers", "connections", "capacity_kw_total", "kwh_per_kw_capacity", "conn_per_trafo"]:
    x = b[f].fillna(b[f].median())
    print(f"  {f:22s} ROC-AUC = {roc_auc_score(y, x):.3f}   PR-AUC = {average_precision_score(y, x):.3f}")

50 features:
['n_transformers', 'capacity_kw_total', 'connections', 'total_kwh', 'kwh_per_conn', 'share_3phase_pct', 'consume_only_pct', 'smart_meter_pct', 'low_tariff_pct', 'a_inw', 'a_hh', 'g_hhgro', 'bev_dich', 'a_woning', 'a_bedv', 'a_opp_ha', 'a_lan_ha', 'ste_mvs', 'ste_oad', 'p_1gezw', 'p_mgezw', 'p_koopw', 'p_huurw', 'p_leegsw', 'p_bjj2k', 'p_bjo2k', 'g_ele', 'g_gas', 'g_ink_pi', 'a_pau', 'g_pau_hh', 'p_bj_me10', 'p_bj_mi10', 'p_won_z', 'p_won_m', 'p_won_zs', 'p_won_ev', 'woz', 'cbs_code_found', 'g_ele_tr', 'temp_mean_c', 'hdd_year', 'cdd_year', 'frost_days', 'solar_kwh_m2_year', 'conn_per_trafo', 'kwh_per_kw_capacity', 'kw_capacity_per_conn', 'solar_homes_per_trafo', 'feedin_kwh_per_kw_cap']

Buurten: 5,423 | municipalities: 141
Stressed share (= PR-AUC of guessing): 0.102

Single-feature baselines:
  n_transformers         ROC-AUC = 0.702   PR-AUC = 0.230
  connections            ROC-AUC = 0.676   PR-AUC = 0.218
  capacity_kw_total      ROC-AUC = 0.702   PR-AUC = 0.234
  kwh_p

In [2]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

params = dict(
    objective="binary", n_estimators=400, learning_rate=0.03,
    num_leaves=15, min_child_samples=30,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=1.0, random_state=42, verbose=-1,
)

def cv_oof(feats, target="stress_nbl"):
    """Out-of-fold predictions: every buurt is predicted by a model
    that never saw its municipality."""
    oof = np.zeros(len(b))
    for tr, te in GroupKFold(n_splits=5).split(b, b[target], groups=b["gm"]):
        m = lgb.LGBMClassifier(**params)
        m.fit(b.iloc[tr][feats], b.iloc[tr][target])
        oof[te] = m.predict_proba(b.iloc[te][feats])[:, 1]
    return oof

size_feats = ["n_transformers", "capacity_kw_total", "connections"]

results = {}
for name, feats in [("Size only (3 features)", size_feats), ("All features (50)", features)]:
    oof = cv_oof(feats)
    results[name] = oof
    print(f"{name:24s} ROC-AUC = {roc_auc_score(y, oof):.3f}   PR-AUC = {average_precision_score(y, oof):.3f}")

Size only (3 features)   ROC-AUC = 0.684   PR-AUC = 0.200
All features (50)        ROC-AUC = 0.753   PR-AUC = 0.271


In [3]:
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr

params_r = {**params, "objective": "regression"}

def cv_oof_reg(feats, target="util_weighted"):
    oof = np.zeros(len(b))
    for tr, te in GroupKFold(n_splits=5).split(b, b[target], groups=b["gm"]):
        m = lgb.LGBMRegressor(**params_r)
        m.fit(b.iloc[tr][feats], b.iloc[tr][target])
        oof[te] = m.predict(b.iloc[te][feats])
    return oof

yr = b["util_weighted"]
print(f"util_weighted: mean {yr.mean():.3f}, std {yr.std():.3f}")
print(f"Baseline (predict the average)   MAE = {mean_absolute_error(yr, np.full(len(yr), yr.mean())):.4f}   R2 = 0.000")

for name, feats in [("Size only (3 features)", size_feats), ("All features (50)", features)]:
    oof = cv_oof_reg(feats)
    results["reg: " + name] = oof
    print(f"{name:32s} MAE = {mean_absolute_error(yr, oof):.4f}   "
          f"R2 = {r2_score(yr, oof):.3f}   ranking = {spearmanr(yr, oof).correlation:.3f}")

util_weighted: mean 0.423, std 0.164
Baseline (predict the average)   MAE = 0.1299   R2 = 0.000
Size only (3 features)           MAE = 0.0969   R2 = 0.369   ranking = 0.628
All features (50)                MAE = 0.0838   R2 = 0.502   ranking = 0.737


In [4]:
import shap

final_clf = lgb.LGBMClassifier(**params).fit(b[features], b["stress_nbl"])
final_reg = lgb.LGBMRegressor(**params_r).fit(b[features], b["util_weighted"])

def shap_table(model, X):
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list):          # some versions return one array per class
        sv = sv[1]
    imp = pd.DataFrame({"feature": X.columns, "mean_abs_shap": np.abs(sv).mean(axis=0)})
    # direction: does a HIGHER feature value push the prediction up (+) or down (-)?
    directions = []
    for i, c in enumerate(X.columns):
        x = X[c]
        mask = x.notna().values
        directions.append(spearmanr(x[mask], sv[mask, i]).correlation if mask.sum() > 10 else np.nan)
    imp["direction"] = directions
    return imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True), sv

imp_clf, sv_clf = shap_table(final_clf, b[features])
imp_reg, sv_reg = shap_table(final_reg, b[features])

print("TOP 15 drivers of stress_nbl (classification):")
print(imp_clf.head(15).round(3).to_string())
print("\nTOP 15 drivers of util_weighted (regression):")
print(imp_reg.head(15).round(3).to_string())

e:\Deloitte 2026\grid_stress_project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Deloitte 2026\grid_stress_project\.venv\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
C:\Users\Karam\AppData\Local\Temp\ipykernel_20928\605953580.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  directions.append(spearmanr(x[mask], sv[mask, i]).correlation if mask.sum() > 10 else np.nan)


TOP 15 drivers of stress_nbl (classification):
                  feature  mean_abs_shap  direction
0               total_kwh          0.368      0.963
1                  a_bedv          0.335      0.940
2          n_transformers          0.272      0.949
3     kwh_per_kw_capacity          0.186      0.893
4                   g_ele          0.127      0.741
5          conn_per_trafo          0.119     -0.883
6                 p_won_z          0.114     -0.832
7   solar_homes_per_trafo          0.112     -0.718
8                bev_dich          0.106     -0.038
9                g_ink_pi          0.095     -0.347
10       share_3phase_pct          0.092     -0.762
11           kwh_per_conn          0.090      0.887
12                ste_oad          0.087      0.634
13                p_bjj2k          0.076      0.340
14  feedin_kwh_per_kw_cap          0.076      0.303

TOP 15 drivers of util_weighted (regression):
                  feature  mean_abs_shap  direction
0     kwh_per_kw_capac

C:\Users\Karam\AppData\Local\Temp\ipykernel_20928\605953580.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  directions.append(spearmanr(x[mask], sv[mask, i]).correlation if mask.sum() > 10 else np.nan)


⚠️ **Superseded.** The next two cells create the **first version** of the driver labels, which contained errors (fixed wording that did not follow the buurt's real values). They are kept for transparency. The corrected labels are made further below (cell with `drivers_for`); those are the ones saved in `buurt_stress_predictions.csv`.

In [5]:
labels = {
    "kwh_per_kw_capacity": "high demand per kW of transformer capacity",
    "kw_capacity_per_conn": "little transformer capacity per connection",
    "bev_dich": "high population density",
    "a_opp_ha": "small area (compact buurt)",
    "a_lan_ha": "small land area (compact buurt)",
    "p_bj_me10": "many homes older than 10 years",
    "p_won_z": "few gas-free homes",
    "share_3phase_pct": "few 3-phase connections",
    "temp_mean_c": "colder location",
    "total_kwh": "high total electricity demand",
    "a_bedv": "many businesses",
    "n_transformers": "many transformers (large buurt)",
    "g_ele": "high electricity use per home",
    "conn_per_trafo": "many connections per transformer",
    "kwh_per_conn": "high demand per connection",
    "ste_oad": "high address density",
}

out = b[["buurtcode", "stress_nbl", "stress_capacity", "util_weighted", "n_transformers"]].copy()
out["risk_prob"] = results["All features (50)"]            # OOF probability of stress_nbl
out["util_pred"] = results["reg: All features (50)"]       # OOF predicted utilisation

# top 3 drivers that push utilisation UP in each buurt (regression SHAP)
sv = pd.DataFrame(sv_reg, columns=features)
top3 = sv.apply(lambda r: r[r > 0].sort_values(ascending=False).index[:3].tolist(), axis=1)
for i in range(3):
    out[f"driver_{i + 1}"] = [labels.get(t[i], t[i]) if len(t) > i else None for t in top3]

# risk tiers based on the OOF probability
out["risk_tier"] = pd.qcut(out["risk_prob"], [0, 0.5, 0.8, 0.95, 1.0],
                           labels=["Low", "Medium", "High", "Very high"])

print(out["risk_tier"].value_counts().sort_index().to_string())
print("\nActual stress rate per tier (%):")
print((out.groupby("risk_tier", observed=True)["stress_nbl"].mean() * 100).round(1).to_string())
print("\nExample rows:")
print(out.sort_values("risk_prob", ascending=False).head(5).to_string(index=False))

out.to_csv(PROCESSED / "buurt_stress_predictions.csv", index=False)
imp_reg.to_csv(PROCESSED / "shap_importance_util.csv", index=False)
imp_clf.to_csv(PROCESSED / "shap_importance_stress.csv", index=False)
print("\nSaved: buurt_stress_predictions.csv, shap_importance_util.csv, shap_importance_stress.csv")

risk_tier
Low          2712
Medium       1626
High          813
Very high     272

Actual stress rate per tier (%):
risk_tier
Low           3.7
Medium       10.8
High         21.8
Very high    37.1

Example rows:
 buurtcode  stress_nbl  stress_capacity  util_weighted  n_transformers  risk_prob  util_pred                                   driver_1                                   driver_2                        driver_3 risk_tier
BU02020100           1                0       0.457834              21   0.849992   0.376310                            many businesses                 small area (compact buurt) small land area (compact buurt) Very high
BU02020102           0                0       0.496548              20   0.770475   0.477312                 small area (compact buurt)            small land area (compact buurt)                 many businesses Very high
BU02281304           0                0       0.474526               9   0.762338   0.429137                            many

⚠️ **Superseded** (see the note above): first-version labels, replaced by the `drivers_for` cell below.

In [6]:
labels.update({
    "a_opp_ha": "compact buurt (small area)",
    "a_lan_ha": "compact buurt (small area)",     # same label -> counted once
    "a_inw": "many inhabitants",
    "a_hh": "many households",
    "a_woning": "many homes",
    "connections": "many connections",
    "capacity_kw_total": "large total capacity (large buurt)",
    "p_mgezw": "many apartments",
    "p_1gezw": "few single-family homes",
    "p_huurw": "many rental homes",
    "woz": "home value",
    "g_ink_pi": "lower income",
    "p_won_zs": "few homes with solar panels",
    "solar_homes_per_trafo": "few solar homes per transformer",
    "feedin_kwh_per_kw_cap": "solar feed-in per kW of capacity",
    "hdd_year": "colder location",
})

def top_drivers(row, n=3):
    chosen = []
    for feat in row[row > 0].sort_values(ascending=False).index:
        lab = labels.get(feat, feat.replace("_", " "))
        if lab not in chosen:                 # skip duplicate labels
            chosen.append(lab)
        if len(chosen) == n:
            break
    return chosen + [None] * (n - len(chosen))

drivers = sv.apply(top_drivers, axis=1, result_type="expand")
out[["driver_1", "driver_2", "driver_3"]] = drivers.values

print("Most common first driver:")
print(out["driver_1"].value_counts().head(10).to_string())
print("\nTop 5 buurten:")
print(out.sort_values("risk_prob", ascending=False)
         [["buurtcode", "risk_tier", "risk_prob", "driver_1", "driver_2", "driver_3"]]
         .head(5).to_string(index=False))

out.to_csv(PROCESSED / "buurt_stress_predictions.csv", index=False)
print("\nSaved again: buurt_stress_predictions.csv")

Most common first driver:
driver_1
high demand per kW of transformer capacity    2172
compact buurt (small area)                     707
many businesses                                636
many connections per transformer               629
little transformer capacity per connection     344
large total capacity (large buurt)             280
high demand per connection                     118
high population density                         71
few 3-phase connections                         62
solar feed-in per kW of capacity                59

Top 5 buurten:
 buurtcode risk_tier  risk_prob                                   driver_1                                   driver_2                         driver_3
BU02020100 Very high   0.849992                            many businesses                 compact buurt (small area)          high population density
BU02020102 Very high   0.770475                 compact buurt (small area)                            many businesses many connections pe

In [7]:
# neutral names: the word "high"/"low" is added from the buurt's REAL value
names_map = {
    "kwh_per_kw_capacity": "demand per kW of transformer capacity",
    "kw_capacity_per_conn": "transformer capacity per connection",
    "conn_per_trafo": "connections per transformer",
    "solar_homes_per_trafo": "solar homes per transformer",
    "feedin_kwh_per_kw_cap": "solar feed-in per kW of capacity",
    "total_kwh": "total electricity demand", "kwh_per_conn": "demand per connection",
    "connections": "number of connections", "n_transformers": "number of transformers",
    "capacity_kw_total": "total transformer capacity",
    "a_bedv": "number of businesses", "a_inw": "number of inhabitants",
    "a_hh": "number of households", "a_woning": "number of homes",
    "bev_dich": "population density", "ste_oad": "address density",
    "a_opp_ha": "buurt area", "a_lan_ha": "buurt area",
    "p_1gezw": "share of single-family homes", "p_mgezw": "share of apartments",
    "p_koopw": "share of owner-occupied homes", "p_huurw": "share of rental homes",
    "p_leegsw": "share of empty homes", "p_bj_me10": "share of homes older than 10 years",
    "p_bj_mi10": "share of homes built in the last 10 years",
    "p_bjj2k": "share of homes built before 2000", "p_bjo2k": "share of homes built after 2000",
    "p_won_z": "share of gas-free homes", "p_won_m": "share of gas-heated homes",
    "p_won_zs": "share of homes with solar panels", "p_won_ev": "share of electrically heated homes",
    "share_3phase_pct": "share of 3-phase connections", "g_ele": "electricity use per home",
    "g_gas": "gas use per home", "g_ele_tr": "solar feed-in per home",
    "consume_only_pct": "share of connections without feed-in",
    "smart_meter_pct": "share of smart meters", "low_tariff_pct": "share of low-tariff use",
    "g_hhgro": "household size", "g_ink_pi": "income per inhabitant", "woz": "home value",
    "a_pau": "number of cars", "g_pau_hh": "cars per household",
    "temp_mean_c": "average temperature", "hdd_year": "heating degree days",
    "cdd_year": "cooling degree days", "frost_days": "frost days",
    "solar_kwh_m2_year": "sunlight",
}
medians = b[features].median()

def drivers_for(i, n=3):
    row = sv.iloc[i]
    chosen, used_names = [], set()
    for feat in row[row > 0].sort_values(ascending=False).index:
        name = names_map.get(feat, feat.replace("_", " "))
        if name in used_names:
            continue
        value = b.iloc[i][feat]
        if pd.isna(value):
            label = f"{name}: unknown (hidden by CBS)"
        else:
            label = f"{'high' if value >= medians[feat] else 'low'} {name}"
        chosen.append((feat, label))
        used_names.add(name)
        if len(chosen) == n:
            break
    return chosen + [(None, None)] * (n - len(chosen))

all_drivers = [drivers_for(i) for i in range(len(b))]
for k in range(3):
    out[f"driver_{k + 1}"] = [d[k][1] for d in all_drivers]
    out[f"driver_{k + 1}_feature"] = [d[k][0] for d in all_drivers]

# CHECK: for every label, is the real value on the side the label says?
wrong = 0
for k in range(3):
    for i, (feat, label) in enumerate(d[k] for d in all_drivers):
        if feat is None or "unknown" in label:
            continue
        above = b.iloc[i][feat] >= medians[feat]
        if label.startswith("high") != above:
            wrong += 1
print("Labels contradicting the real value:", wrong)

print("\nMost common first driver:")
print(out["driver_1"].value_counts().head(12).to_string())

out.to_csv(PROCESSED / "buurt_stress_predictions.csv", index=False)
print("\nSaved: buurt_stress_predictions.csv (corrected labels)")

Labels contradicting the real value: 0

Most common first driver:
driver_1
high demand per kW of transformer capacity    2170
low buurt area                                 667
high number of businesses                      635
low connections per transformer                619
low transformer capacity per connection        344
low total transformer capacity                 280
high demand per connection                     118
low share of 3-phase connections                62
high solar feed-in per kW of capacity           59
low total electricity demand                    56
low population density                          44
high electricity use per home                   41

Saved: buurt_stress_predictions.csv (corrected labels)


In [8]:
from scipy.stats import spearmanr

PV = Path("../Data/raw/4 Liander/Raw Data/4 Generation data for small-scale connections/"
          "Liander_decentrale_opwek_kv_zon_20240409.csv")
pv = pd.read_csv(PV, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
pv.columns = pv.columns.str.strip()
pv["buurtcode"] = pv["CBS_BUURTCODE_2023"].str.strip()
print("Rows hidden for privacy ('Onbepaald'):", (~pv["buurtcode"].str.startswith("BU", na=False)).sum())
pv = pv[pv["buurtcode"].str.startswith("BU", na=False)]
for c in ["AANTAL_AANSLUITINGEN_MET_PV", "TOTAAL_AANTAL_AANSLUITINGEN", "OPGESTELD_VERMOGEN_KW"]:
    pv[c] = pd.to_numeric(pv[c].str.strip().str.replace(",", ".", regex=False), errors="coerce")

# min_count=1: a buurt with only empty values stays EMPTY instead of becoming 0
pvb = pv.groupby("buurtcode").agg(
    pv_connections=("AANTAL_AANSLUITINGEN_MET_PV", lambda s: s.sum(min_count=1)),
    pv_total_conn=("TOTAAL_AANTAL_AANSLUITINGEN", lambda s: s.sum(min_count=1)),
    pv_kw=("OPGESTELD_VERMOGEN_KW", lambda s: s.sum(min_count=1)),
).reset_index()

q = b.merge(pvb, on="buurtcode", how="left")
print("Model B buurten:", len(q))
print("  with PV value > 0:", (q["pv_kw"] > 0).sum())
print("  with PV value = 0:", (q["pv_kw"] == 0).sum())
print("  with no PV value: ", q["pv_kw"].isna().sum())

q["pv_kw_per_kw_cap"] = q["pv_kw"] / q["capacity_kw_total"]
q["avg_load_kw_per_kw_cap"] = (q["total_kwh"] / 8760) / q["capacity_kw_total"]

# tests only on buurten with a real (> 0) PV value
r = q[q["pv_kw"] > 0]
print(f"\nPeak (kW) vs installed solar (kW):   Spearman {spearmanr(r['peak_kw_total'], r['pv_kw']).correlation:.3f}")
print(f"Peak (kW) vs yearly consumption:      Spearman {spearmanr(r['peak_kw_total'], r['total_kwh']).correlation:.3f}")
print(f"Median peak / installed solar:        {(r['peak_kw_total'] / r['pv_kw']).median():.2f}")
print(f"\nUtilisation vs solar kW per kW cap:   Spearman {spearmanr(r['util_weighted'], r['pv_kw_per_kw_cap']).correlation:.3f}")
print(f"Utilisation vs avg load per kW cap:   Spearman {spearmanr(r['util_weighted'], r['avg_load_kw_per_kw_cap']).correlation:.3f}")

# groups: zeros separately, the rest in 4 equal groups
q["group"] = "0 kW (or hidden)"
pos = q["pv_kw"] > 0
q.loc[pos, "group"] = pd.qcut(q.loc[pos, "pv_kw_per_kw_cap"], 4,
                              labels=["Q1 low", "Q2", "Q3", "Q4 high"]).astype(str)
q.loc[q["pv_kw"].isna(), "group"] = "no PV data"
g = q.groupby("group").agg(
    buurten=("buurtcode", "size"),
    pv_per_kw_cap=("pv_kw_per_kw_cap", "median"),
    avg_util=("util_weighted", "mean"),
    stress_rate_pct=("stress_nbl", "mean"),
)
g["stress_rate_pct"] *= 100
print("\nBy solar kW per kW of transformer capacity:")
print(g.round(3).to_string())

Rows hidden for privacy ('Onbepaald'): 2284
Model B buurten: 5423
  with PV value > 0: 3143
  with PV value = 0: 0
  with no PV value:  2280

Peak (kW) vs installed solar (kW):   Spearman 0.435
Peak (kW) vs yearly consumption:      Spearman 0.795
Median peak / installed solar:        0.97

Utilisation vs solar kW per kW cap:   Spearman 0.355
Utilisation vs avg load per kW cap:   Spearman 0.569

By solar kW per kW of transformer capacity:
            buurten  pv_per_kw_cap  avg_util  stress_rate_pct
group                                                        
Q1 low          786          0.108     0.343           12.087
Q2              786          0.270     0.363            6.616
Q3              785          0.471     0.407            5.223
Q4 high         786          0.902     0.506            4.453
no PV data     2280            NaN     0.447           14.561


In [9]:
# 1. share of summer peaks per solar group
g2 = q.groupby("group").agg(
    buurten=("buurtcode", "size"),
    share_summer_peaks=("share_summer_peaks", "mean"),
)
print("Share of transformer peaks in April-September, by solar group:")
print((g2 * [1, 100]).round(1).to_string())

# 2. what do buurten WITHOUT PV data look like?
q["pv_missing"] = q["pv_kw"].isna()
prof = q.groupby("pv_missing").agg(
    buurten=("buurtcode", "size"),
    median_connections=("connections", "median"),
    very_urban_pct=("ste_mvs", lambda s: (s == 1).mean() * 100),
    cbs_solar_homes_pct=("p_won_zs", "median"),
    median_capacity_kw=("capacity_kw_total", "median"),
)
print("\nProfile of buurten with and without Liander PV data:")
print(prof.round(1).to_string())

Share of transformer peaks in April-September, by solar group:
            buurten  share_summer_peaks
group                                  
Q1 low          786                97.9
Q2              786                98.4
Q3              785                98.3
Q4 high         786                98.5
no PV data     2280                99.0

Profile of buurten with and without Liander PV data:
            buurten  median_connections  very_urban_pct  cbs_solar_homes_pct  median_capacity_kw
pv_missing                                                                                      
False          3143               239.0            11.0                 41.0              1110.0
True           2280               772.5            20.8                 42.0              2110.0
